In [ ]:
REPO_URL = "https://github.com/krpchandok/ml-experimentation-platform.git"
BRANCH = "main"
!nvidia-smi
!nproc && free -g && uname -r && cat /proc/self/cgroup && cat /proc/sys/kernel/task_delayacct

In [ ]:
import os
if not os.path.isdir("/content/ml-experimentation-platform"):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/ml-experimentation-platform
%cd /content/ml-experimentation-platform
!git log --oneline -1

In [ ]:
!cmake -S agent -B agent/build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build agent/build -j 4 2>&1 | tail -3
!ldd agent/build/mlplat-agent | grep -c nvidia || echo "agent has no link-time NVIDIA dependency"
!agent/build/mlplat_agent_tests | tail -1
!pip install -q -e . pytest
!python -m pytest agent/tests/integration -q -rs 2>&1 | tail -5

In [ ]:
!PYTHON=python PREFIX=colab STEPS=1000 PREPROCESS_COST=6 WORKER_COUNTS="0 2" bash examples/gpu_demo.sh 2>&1 | grep -v '^mlplat: run 2'

In [ ]:
!python -m mlplat run --name colab-workers-2-log10 -- python examples/train_image_classifier.py --device cuda --steps 2000 --preprocess-cost 6 --workers 2 --log-every 10 > /dev/null
!python examples/summarize_runs.py --prefix colab

In [ ]:
import json, glob
for path in sorted(glob.glob("runs/*/resources.jsonl")):
    with open(path) as handle:
        lines = [json.loads(line) for line in handle]
    header, end = lines[0], lines[-1]
    gpu_samples = [line["gpu"] for line in lines if line["type"] == "sample" and isinstance(line["gpu"], dict)]
    print(path.split("/")[1], json.dumps(header["gpu"])[:300])
    print("  process_info:", any(sample["process_info"] for sample in gpu_samples),
          "tracked pids on GPU:", sorted({proc["pid"] for sample in gpu_samples for proc in sample["procs"]}),
          "unavailable:", sorted({name for sample in gpu_samples for device in sample["devices"] for name in device.get("unavailable", [])}))
    print("  agent:", {key: end.get(key) for key in ("agent_cpu_pct", "sample_us_mean", "startup_cpu_s", "agent_max_rss_kb")})

In [ ]:
!tar czf /content/colab_runs.tgz runs
from google.colab import files
files.download("/content/colab_runs.tgz")